In [1]:
import pandas as pd 
import os 
import logging 
import traceback
import re
import ezodf
from pathlib import Path
from basicprocess import create_folder, outputlog, findfiles, read_combined_dataframe
from TDXdataframe import read_businfo_xml
from openpyxl import load_workbook


In [ ]:
def get_taipeibusreport(odspath):

    logging.info("開始讀取台北市營運月報")
    logging.info(f"台北市公車營運月報的原始檔案為{odspath}")

    sheet_names = pd.ExcelFile(odspath).sheet_names
    dfs = []
    for sheet in sheet_names:
        odsdf = pd.read_excel(odspath, sheet_name=sheet, engine='odf')
        odsdf['Time'] = sheet
        dfs.append(odsdf)

    df = pd.concat(dfs, ignore_index=True)

    logging.info("台北市營運月報整併完成")
    

    df['年'] = df['Time'].str[:3].astype(int) + 1911
    df['月'] = df['Time'].str[3:].astype(int)
    # df['資料時間'] = pd.to_datetime(df['年'].astype(str) + '-' + df['月'].astype(str)).dt.to_period('M')
    df['資料時間'] = pd.to_datetime(
        df['年'].astype(str) + '-' + df['月'].astype(str).str.zfill(2),
        format='%Y-%m',
        errors='raise').dt.to_period('M')

    cols = ['資料時間'] + [c for c in df.columns if c not in ['年', '月', 'Time', '資料時間']]
    df = df.reindex(columns=cols)

    logging.info("輸出指定格式")

    rename_dist = {'資料時間':'Month',
                '客運業者':'OperatorName', 
                '路線代碼':'RouteID', 
                '路線別':'RouteName', 
                '總班次':'Shifts', 
                '總載客人次':'Passenger',
                '總行駛里程':'Miles',
                '延人公里':'PaxKm', 
                '總營收': 'Revenue'}
    df = df[list(rename_dist)]
    df = df.rename(columns = rename_dist)

    return df

def check_if_morethanone(df, checklists, timecolumn, warningfolder, dataname = '資料'):
    """
    檢查每個 timecolumn 內，checklists 是否有重複值

    Parameters
    ----------
    df : pandas.DataFrame
    checklists : list
        需要檢查是否重複的欄位
    timecolumn : str
        時間欄位（例如 Month）

    Returns
    -------
    duplicated_df : pandas.DataFrame
        含有重複資料的 dataframe（只保留重複者）
    summary : pandas.DataFrame
        每個月份重複筆數的摘要
    """

    # 找出在「同一個月 + checklists」下重複的資料
    mask = df.duplicated(subset=[timecolumn] + checklists, keep=False)
    duplicated_df = df[mask].sort_values([timecolumn] + checklists)

    if len(duplicated_df) > 0:
        logging.warning(f"{dataname} 有重複的資料")

        # 每月重複筆數摘要
        summary = (
            duplicated_df
            .groupby(timecolumn)
            .size()
            .reset_index(name='DuplicatedRows')
        )

        outputfile = os.path.join(warningfolder, f"{dataname}重複資料.xlsx")

        with pd.ExcelWriter(outputfile, engine='xlsxwriter') as writer:
            duplicated_df.to_excel(writer, index=True, sheet_name='有重複的資料')
            summary.to_excel(writer, index=True, sheet_name='每月重複筆數')

        logging.info(f"{dataname}路線營運月報，路徑：{outputfile}")

        return False     

    else:
        logging.info(f"{dataname}在{checklists}的組合底下沒有重複資料")
        return True

def get_businfo():
    businfos = []
    for xmlpath in findfiles(filefolderpath=os.path.join(os.getcwd(), '..', '00_TDX資料下載', '03公車路線營運資料'), filetype='xml'):
        businfo = read_businfo_xml(xml_path=xmlpath)
        businfos.append(businfo)
    businfo = pd.concat(businfos)

    return businfo

def flatten_columns(multi_cols):
    flat_cols = []
    for col in multi_cols:
        parts = [
            str(v)
            for v in col
            if v is not None and not pd.isna(v) and str(v).strip() != ""
        ]
        flat_cols.append("_".join(parts))
    return flat_cols

def simplify_col(c: str) -> str:
    # 1. 移除單位 (括號內)
    c = re.sub(r"_?\([^)]*\)", "", c)

    # 2. 類別縮寫
    c = c.replace("路線營運資料_", "路線_")
    c = c.replace("車輛情形_", "車輛_")
    c = c.replace("包車出租_", "包車_")

    # 3. 常見冗字精簡
    c = c.replace("營業行車次數", "行車次數")
    c = c.replace("營業行駛里程", "行駛里程")
    c = c.replace("營業里程", "營業里程")
    c = c.replace("行駛延日車數", "延日車數")

    # 4. 多餘底線清理
    c = re.sub(r"__+", "_", c).strip("_")

    return c

def read_specific_data(filepath, sheetname, cell):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".xlsx":
        wb = load_workbook(filepath, data_only=True)
        ws = wb[sheetname]
        return ws[cell].value

    elif ext == ".ods":
        doc = ezodf.opendoc(filepath)
        sheet = doc.sheets[sheetname]
        col = ord(cell[0].upper()) - ord('A')
        row = int(cell[1:]) - 1
        return sheet[row, col].value

    else:
        raise ValueError("不支援的檔案格式")

def get_excel_sheet_names(path):
    """
    取得 Excel 檔案中的所有工作表名稱。

    Args:
        path (str): Excel 檔案的路徑。

    Returns:
        list: 工作表名稱列表。
    """
    try:
        sheet_names = pd.ExcelFile(path).sheet_names
        return sheet_names
    except FileNotFoundError:
        print(f"檔案不存在：{path}")
        return []
    except Exception as e:
        print(f"發生錯誤：{e}")
        return []
    
def combined_all_bus_monthlyreport(folder, countyname = False):
    if countyname != False:
        logging.info(f"開始讀取{countyname}縣市不分路線營運月報")
    else:
        logging.warning(f"讀取未知縣市的不分路線營運月報")

    dfs = []
    files = findfiles(folder, '', recursive=False)
    for file in files :
        sheetnames = get_excel_sheet_names(file)
        if '2522-02-01-2' in sheetnames:
            sheetname = '2522-02-01-2'
        else:
            sheetname = sheetnames[0]
        timetext = read_specific_data(file, sheetname, 'A4')
        m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
        dyear = int(m.group(1))
        dmonth = int(m.group(2))
        df = pd.read_excel(file, skiprows=4, header = [0, 1, 2, 3, 4])
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
                for v in col
            )
            for col in df.columns
        )
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None
                if (
                    pd.isna(v)
                    or (isinstance(v, str) and v.startswith("Unnamed:"))
                )
                else v
                for v in col
            )
            for col in df.columns
        )
        df.columns = flatten_columns(df.columns)
        df.columns = [simplify_col(c) for c in df.columns]
        df['市區客運業者家數'] = pd.to_numeric(df['市區客運業者家數'], errors='coerce')
        df = df[(~df['市區客運業者家數'].isna()) & (df['項目'] == '總計')]
        df['年'] = dyear
        df['月'] = dmonth

        cols = df.columns.tolist()
        new_cols = ['年', '月'] + [c for c in cols if c not in ['年', '月']]
        df = df[new_cols]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    if countyname != False:
        df['縣市'] = countyname
        cols = df.columns.tolist()
        new_cols = ['年', '月', '縣市'] + [c for c in cols if c not in ['年', '月', '縣市']]
        df = df[new_cols]


    return df.sort_values(['年', '月']).reset_index(drop = True)

def organized_bus_info(referencefolder):
    logging.info('開始整併計算市區公車每條路線的人次')
    filepath = os.path.join(referencefolder, '月報.xlsx')
    df_bus = pd.read_excel(filepath, sheet_name='市區公車')
    df_bus = df_bus.iloc[1:]
    df_bus.columns = ['縣市', '年', '月', '客運業者', '路線名稱', '起始站', '終點站', '營業里程', '座位數', '班次', 'X', '行駛里程', '人次', '延人公里', '收入', 'XX', 'XXX', 'XXXXX']
    df_bus = df_bus.loc[:, ~df_bus.columns.str.contains('X')]

    df_bus['年'] = pd.to_numeric(df_bus['年'], errors='coerce').astype('int64')
    df_bus['年'] = df_bus['年'] + 1911
    df_bus['月'] = pd.to_numeric(df_bus['月'], errors='coerce').astype('int64')
    df_bus['Month'] = pd.PeriodIndex(year=df_bus['年'],
                                    month=df_bus['月'],
                                    freq='M')
    bus_groupbycolumns = ['縣市', 'Month', '路線名稱']
    df_bus = df_bus.groupby(bus_groupbycolumns).agg({'人次':'sum'}).reset_index()
    df_bus['縣市'] = df_bus['縣市'].map({
                                        '臺北市': 'TPE',
                                        '新北市': 'NWT',
                                        '基隆市': 'KEE',
                                        '桃園市': 'TAO'})
    
    return df_bus

def organized_intercitybus_info(referencefolder):
    logging.info('開始整併計算公路客運每條路線的人次')
    filepath = os.path.join(referencefolder, '月報.xlsx')
    df_bus = pd.read_excel(filepath, sheet_name='公路客運')

    df_bus['年'] = pd.to_numeric(df_bus['年'], errors='coerce').astype('int64')
    df_bus['年'] = df_bus['年'] + 1911
    df_bus['月'] = pd.to_numeric(df_bus['月'], errors='coerce').astype('int64')
    df_bus['Month'] = pd.PeriodIndex(year=df_bus['年'],
                                    month=df_bus['月'],
                                    freq='M')


    bus_groupbycolumns = [ 'Month', '路線編號']
    df_bus = df_bus.groupby(bus_groupbycolumns).agg({'客運人數(人次)':'sum'}).reset_index()

    df_bus.columns = ['Month', '路線名稱', '人次']

    return df_bus



In [3]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '04_營運月報整理.log'))
if os.path.exists(logfile):
    os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

def main():
    logging.info('Start Processing...')

    referencefolder = os.path.join(os.getcwd(), '..', '參考資料')
    outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
    monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))

    monthlyreport_organized_bycounty_folder = create_folder(os.path.join(monthlyreport_organized_folder, '01_不分路線'))
    monthlyreport_organized_byroute_folder = create_folder(os.path.join(monthlyreport_organized_folder, '02_區分路線'))
    
    finalorganizedfile = os.path.join(monthlyreport_organized_byroute_folder, '月報統計數量.xlsx')
    
    # 處理台北公車路線資料
    # (1) 沒有區分路線的


    # (2) 有區分路線的 (一開始鎮昌處理的，但後來沒有要處理了)
    
    # taipeidf = get_taipeibusreport(r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\04臺北\01公運處\附2-項目(二)臺北市營運資料(市區公車)v1.ods")
    # taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
    # logging.info('輸出臺北市公車營運月報整理結果')
    # temp = check_if_morethanone(df = taipeidf, 
    #                             checklists = ['RouteID', 'RouteName' ], 
    #                             timecolumn = 'Month', 
    #                             dataname = '臺北市公車營運月報', 
    #                             warningfolder=create_folder(os.path.join(monthlyreport_organized_byroute_folder, 'error')))
    # if temp == False:
    #     logging.info("臺北市公車因為有多個OpeartionName經營同一條路線")
    #     taipeidf = taipeidf.drop(columns='OperatorName').groupby(['Month','RouteName']).agg({'Shifts':'sum',
    #                                                                                          'Passenger':'sum',
    #                                                                                          'Miles':'sum',
    #                                                                                          'PaxKm':'sum',
    #                                                                                          'Revenue':'sum'}).reset_index()
    #     taipeidf.to_excel(os.path.join(monthlyreport_organized_byroute_folder, '臺北市公車營運月報.xlsx'), index = False)
    #     logging.info('重新輸出臺北市公車營運月報整理結果')

    # del temp

    # 處理桃園公車路線資料
    # (1) 沒有區分路線的
    taoyuan_df = combined_all_bus_monthlyreport(folder=r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\06桃園\02桃園市區客運業者營運概況", 
                                                countyname='桃園市')
    taoyuan_df.to_excel(os.path.join(monthlyreport_organized_bycounty_folder, '桃園市營運月報_不分路線.xlsx'))
    logging.info(f"桃園不分路線營運月報經整併至 {os.path.join(monthlyreport_organized_bycounty_folder, '桃園市營運月報_不分路線.xlsx')}")
    

    # 處理新北市公車路線資料
    # (1) 沒有區分路線的
    newtaipei_df = combined_all_bus_monthlyreport(folder=r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\05新北\01新北市區客運業者營運概況\不分路線", 
                                                  countyname='新北市')
    newtaipei_df.to_excel(os.path.join(monthlyreport_organized_bycounty_folder, '新北市營運月報_不分路線.xlsx'))
    logging.info(f"桃園不分路線營運月報經整併至 {os.path.join(monthlyreport_organized_bycounty_folder, '新北市營運月報_不分路線.xlsx')}")



    # 整併所有縣市的分路線資料
    # alldf = read_combined_dataframe(findfiles(monthlyreport_organized_byroute_folder, 'xlsx', recursive=False))
    # alldf.to_excel(finalorganizedfile)
    df_citybus = organized_bus_info(referencefolder=referencefolder)
    df_intercitybus = organized_intercitybus_info(referencefolder=referencefolder)
    df_intercitybus['縣市'] = 'THB'
    pd.concat([df_citybus, df_intercitybus]).to_excel(finalorganizedfile)
    logging.info(f"資料輸出至{finalorganizedfile}")


    logging.info('Finished Processing.')


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)

C:\Users\kjchang\AppData\Local\Temp\ipykernel_4416\3869944720.py:244: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  df_bus['Month'] = pd.PeriodIndex(year=df_bus['年'],
C:\Users\kjchang\AppData\Local\Temp\ipykernel_4416\3869944720.py:265: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  df_bus['Month'] = pd.PeriodIndex(year=df_bus['年'],


In [ ]:
# folder = r"C:\Users\kjchang\Downloads\OneDrive_4_2026-1-8\桃園市區公車11310-11312" 
# countyname = '桃園市'
# errorfile = []
# dfs = []
# files = findfiles(folder, '')

# for file in files:
#     print(file)
#     sheetnames = get_excel_sheet_names(file)

#     if '報表程式' not in sheetnames:
#         errorfile.append(file)
#         print('X')
#         continue

#     sheetname = '報表程式'
#     timetext = read_specific_data(file, sheetname, 'A1')
#     m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
#     dyear = int(m.group(1))
#     dmonth = int(m.group(2))

#     doperator = os.path.basename(file)
#     doperator = os.path.splitext(os.path.basename(file))[0]  # 去副檔名
#     doperator = re.sub(r'\d+', '', doperator)                 # 去數字

#     df = pd.read_excel(file, skiprows=2, header=[0, 1, 2, 3, 4])

#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
#             for v in col
#         )
#         for col in df.columns
#     )
#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None
#             if (
#                 pd.isna(v)
#                 or (isinstance(v, str) and v.startswith("Unnamed:"))
#             )
#             else v
#             for v in col
#         )
#         for col in df.columns
#     )

#     df.columns = flatten_columns(df.columns)
#     df.columns = [simplify_col(c) for c in df.columns]

#     df = df.loc[:, ~df.columns.duplicated()].copy()

#     df['路線_營業里程'] = pd.to_numeric(df['路線_營業里程'], errors='coerce')
#     df = df[(~df['路線_營業里程'].isna()) &
#             (df['路線_核定路線數'] != '合計') &
#             (df['路線_核定路線數'] != '總計')]

#     df['年'] = dyear
#     df['月'] = dmonth
#     df['公司名稱'] = doperator

#     cols = df.columns.tolist()
#     new_cols = ['年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

#     dfs.append(df)

# # ★重要：concat 要在 for 外面，且不要再 flatten/simplify
# df = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

# if countyname is not False and not df.empty:
#     df['縣市'] = countyname
#     cols = df.columns.tolist()
#     new_cols = [ '縣市', '年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

# ===========================

# folder = r"C:\Users\kjchang\Downloads\OneDrive_4_2026-1-8\桃園市區公車11310-11312" 
# countyname = '桃園市'
# errorfile = []
# dfs = []
# files = findfiles(folder, '')

# for file in files:
#     print(file)
#     sheetnames = get_excel_sheet_names(file)

#     if '報表程式' not in sheetnames:
#         errorfile.append(file)
#         print('X')
#         continue

#     sheetname = '報表程式'
#     timetext = read_specific_data(file, sheetname, 'A1')
#     m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
#     dyear = int(m.group(1))
#     dmonth = int(m.group(2))

#     doperator = os.path.basename(file)
#     doperator = os.path.splitext(os.path.basename(file))[0]  # 去副檔名
#     doperator = re.sub(r'\d+', '', doperator)                 # 去數字

#     df = pd.read_excel(file, skiprows=2, header=[0, 1, 2, 3, 4])

#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
#             for v in col
#         )
#         for col in df.columns
#     )
#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None
#             if (
#                 pd.isna(v)
#                 or (isinstance(v, str) and v.startswith("Unnamed:"))
#             )
#             else v
#             for v in col
#         )
#         for col in df.columns
#     )

#     df.columns = flatten_columns(df.columns)
#     df.columns = [simplify_col(c) for c in df.columns]

#     df = df.loc[:, ~df.columns.duplicated()].copy()

#     df['路線_營業里程'] = pd.to_numeric(df['路線_營業里程'], errors='coerce')
#     df = df[(~df['路線_營業里程'].isna()) &
#             (df['路線_核定路線數'] != '合計') &
#             (df['路線_核定路線數'] != '總計')]

#     df['年'] = dyear
#     df['月'] = dmonth
#     df['公司名稱'] = doperator

#     cols = df.columns.tolist()
#     new_cols = ['年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

#     dfs.append(df)

# # ★重要：concat 要在 for 外面，且不要再 flatten/simplify
# df = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

# if countyname is not False and not df.empty:
#     df['縣市'] = countyname
#     cols = df.columns.tolist()
#     new_cols = [ '縣市', '年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

# ===========================

# folder = r"C:\Users\kjchang\Downloads\OneDrive_4_2026-1-8\桃園市區公車11310-11312" 
# countyname = '桃園市'
# errorfile = []
# dfs = []
# files = findfiles(folder, '')

# for file in files:
#     print(file)
#     sheetnames = get_excel_sheet_names(file)

#     if '報表程式' not in sheetnames:
#         errorfile.append(file)
#         print('X')
#         continue

#     sheetname = '報表程式'
#     timetext = read_specific_data(file, sheetname, 'A1')
#     m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
#     dyear = int(m.group(1))
#     dmonth = int(m.group(2))

#     doperator = os.path.basename(file)
#     doperator = os.path.splitext(os.path.basename(file))[0]  # 去副檔名
#     doperator = re.sub(r'\d+', '', doperator)                 # 去數字

#     df = pd.read_excel(file, skiprows=2, header=[0, 1, 2, 3, 4])

#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
#             for v in col
#         )
#         for col in df.columns
#     )
#     df.columns = pd.MultiIndex.from_tuples(
#         tuple(
#             None
#             if (
#                 pd.isna(v)
#                 or (isinstance(v, str) and v.startswith("Unnamed:"))
#             )
#             else v
#             for v in col
#         )
#         for col in df.columns
#     )

#     df.columns = flatten_columns(df.columns)
#     df.columns = [simplify_col(c) for c in df.columns]

#     df = df.loc[:, ~df.columns.duplicated()].copy()

#     df['路線_營業里程'] = pd.to_numeric(df['路線_營業里程'], errors='coerce')
#     df = df[(~df['路線_營業里程'].isna()) &
#             (df['路線_核定路線數'] != '合計') &
#             (df['路線_核定路線數'] != '總計')]

#     df['年'] = dyear
#     df['月'] = dmonth
#     df['公司名稱'] = doperator

#     cols = df.columns.tolist()
#     new_cols = ['年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

#     dfs.append(df)

# # ★重要：concat 要在 for 外面，且不要再 flatten/simplify
# df = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

# if countyname is not False and not df.empty:
#     df['縣市'] = countyname
#     cols = df.columns.tolist()
#     new_cols = [ '縣市', '年', '月', '公司名稱'] + [c for c in cols if c not in ['年', '月', '公司名稱']]
#     df = df[new_cols]

# ===========================
# ['C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11310亞通.xlsx',
#  'C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11310指南.xlsx',
#  'C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11311亞通.xlsx',
#  'C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11311指南.xlsx',
#  'C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11312亞通.xlsx',
#  'C:\\Users\\kjchang\\Downloads\\OneDrive_4_2026-1-8\\桃園市區公車11310-11312\\11312指南.xlsx']


In [ ]:
def combined_all_bus_monthlyreport_keelung(folder, countyname = False):
    if countyname != False:
        logging.info(f"開始讀取{countyname}縣市不分路線營運月報")
    else:
        logging.warning(f"讀取未知縣市的不分路線營運月報")

    dfs = []
    files = findfiles(folder, 'xlsx', recursive=False)
    for file in files :
        sheetnames = get_excel_sheet_names(file)
        if '2522-02-01-2' in sheetnames:
            sheetname = '2522-02-01-2'
        else:
            sheetname = sheetnames[0]
        timetext = read_specific_data(file, sheetname, 'A4')
        m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
        dyear = int(m.group(1))
        dmonth = int(m.group(2))
        df = pd.read_excel(file, skiprows=4, header = [0, 1, 2, 3, 4])
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
                for v in col
            )
            for col in df.columns
        )
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None
                if (
                    pd.isna(v)
                    or (isinstance(v, str) and v.startswith("Unnamed:"))
                )
                else v
                for v in col
            )
            for col in df.columns
        )
        df.columns = flatten_columns(df.columns)
        df.columns = [simplify_col(c) for c in df.columns]
        df['市區客運業者家數'] = pd.to_numeric(df['市區客運業者家數'], errors='coerce')
        df = df[(~df['市區客運業者家數'].isna()) & (df['項目'] == '總計')]
        df['年'] = dyear
        df['月'] = dmonth

        cols = df.columns.tolist()
        new_cols = ['年', '月'] + [c for c in cols if c not in ['年', '月']]
        df = df[new_cols]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    if countyname != False:
        df['縣市'] = countyname
        cols = df.columns.tolist()
        new_cols = ['年', '月', '縣市'] + [c for c in cols if c not in ['年', '月', '縣市']]
        df = df[new_cols]


    return df.sort_values(['年', '月']).reset_index(drop = True)

In [8]:
combined_all_bus_monthlyreport(folder=r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\03基隆營運月報表", 
                               countyname='基隆市')

InvalidIndexError: Reindexing only valid with uniquely valued Index objects